# 04 — Survey-weight sensitivity (standalone check)
The Limitations slide says weighted and unweighted results were generally similar, with **upper
secondary the most weight-sensitive**. This is the code behind that sentence (ported from model
v1, where it originally lived): adoption unweighted vs TCHWGT-weighted, overall and by ISCED level.

Standalone and fast — loads only the three columns it needs from the merged CSV.

In [ ]:
# ============================================================
# CELL 0 — paths (portable: works from the repo root OR from Model/)
# All reads/writes below are relative to these three lines.
# ============================================================
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "Data").exists() else Path.cwd().parent
DATA_DIR = ROOT / "Data"                 # codebook + small CSVs
SPSS_DIR = DATA_DIR / "SPSS"             # raw TALIS .sav files (gitignored)
OUT_DIR  = DATA_DIR / "output"           # everything the notebooks produce (gitignored)
OUT_DIR.mkdir(parents=True, exist_ok=True)
assert DATA_DIR.exists(), f"Data folder not found — start Jupyter from the repo root or Model/ (looked at {DATA_DIR})"

In [ ]:
# ============================================================
# CELL 1 — load just TT4G36 / TCHWGT / IDPOP from the merged file
# (usecols callable -> reads 3 columns, not all 1.6 GB)
# ============================================================
import numpy as np
import pandas as pd

WANT = {'TT4G36', 'TCHWGT', 'IDPOP'}
d = pd.read_csv(OUT_DIR / "teacher_principal_named_columns.csv",
                encoding="utf-8-sig", low_memory=False,
                usecols=lambda c: c.split(' ')[0] in WANT)
d.columns = [c.split(' ')[0] for c in d.columns]
print(f"rows: {len(d):,}  |  columns: {list(d.columns)}")

In [ ]:
# ============================================================
# CELL 2 — weighted vs unweighted adoption (TCHWGT), by ISCED
# Ported from model.ipynb (v1) cells 48-51.
# ============================================================
d = d.rename(columns={'TT4G36': 'q36', 'TCHWGT': 'w', 'IDPOP': 'pop'})
d['q36'] = pd.to_numeric(d['q36'], errors='coerce')
d = d[d['q36'].isin([1, 2])]                    # valid yes/no only
d['y'] = (d['q36'] == 1).astype(int)
d['w'] = pd.to_numeric(d['w'], errors='coerce')
d = d.dropna(subset=['w'])

wmean = lambda s: (s['y'] * s['w']).sum() / s['w'].sum() * 100

print(f"OVERALL:    unweighted {d['y'].mean()*100:5.1f}%  |  weighted {wmean(d):5.1f}%  "
      f"|diff| {abs(d['y'].mean()*100 - wmean(d)):.1f} pp")
print()
for lvl, name in [(1, 'Primary'), (2, 'Lower sec'), (3, 'Upper sec')]:
    sub = d[pd.to_numeric(d['pop'], errors='coerce') == lvl]
    if not len(sub):
        continue
    unw, wtd = sub['y'].mean()*100, wmean(sub)
    print(f"{name:10s}: unweighted {unw:5.1f}%  |  weighted {wtd:5.1f}%  "
          f"|diff| {abs(unw-wtd):.1f} pp   (n={len(sub):,})")
print("\nexpectation from the slide: diffs small everywhere, largest at upper secondary.")